In [1]:
import oxbow as ox

import polars as pl
import sqlite3
import numpy as np
import pandas as pd
from tqdm import tqdm
from sqlitedict import SqliteDict
from collections import defaultdict

In [8]:
!rm anmol.sql
dct = SqliteDict("anmol.sql",autocommit=True,)
dct["description"] = 'NULL'
dct["desc"] = 'NULL'
dct["cutadapt_removed"] =0 

In [2]:
sqlite_file = "./homo_sapiens.Gencode_v25.sqlite"
conn = sqlite3.connect(sqlite_file)

In [3]:
transcriptome = pl.read_database(
    query="SELECT * FROM transcripts", 
    connection=conn,
).filter(pl.col("transcript").is_in(list(set(samfile_mapped["transcript"])))).select(["transcript","cds_start","cds_stop","length","strand","chrom","tran_type"])#.filter(pl.col("tran_type")==1)

NameError: name 'samfile_mapped' is not defined

In [2]:
%%time


arrow_ipc = ox.from_bam("Galaxy93.bam",
                        fields=["qname","rname", "pos","seq"]
                        ,tag_defs=[('NM', 'i'), ('MD', 'Z')]).to_dask()

CPU times: user 3.69 s, sys: 750 ms, total: 4.44 s
Wall time: 3.94 s


In [3]:
arrow_ipc.head()

,qname,rname,pos,seq,tags
0,SRR5882586.1_CAATA,NaN,NaN,CGACTGAGTCCGCGATGGAGAGAGCTGTAGGCACCA,"{'NM': None, 'MD': None}"
1,SRR5882586.2_TAGGC,ENST00000592665.1|ENSG00000130159.13|OTTHUMG00...,298.0,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,"{'NM': 1.0, 'MD': '35A0'}"
2,SRR5882586.2_TAGGC,ENST00000590480.1|ENSG00000130159.13|OTTHUMG00...,501.0,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,"{'NM': 1.0, 'MD': '35A0'}"
3,SRR5882586.2_TAGGC,ENST00000252440.11|ENSG00000130159.13|OTTHUMG0...,410.0,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,"{'NM': 1.0, 'MD': '35A0'}"
4,SRR5882586.2_TAGGC,ENST00000270517.11|ENSG00000130159.13|OTTHUMG0...,447.0,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,"{'NM': 1.0, 'MD': '35A0'}"


In [6]:
%%time
len(arrow_ipc)

CPU times: user 39 s, sys: 3.78 s, total: 42.7 s
Wall time: 47.1 s


11791325

In [7]:
%%time

a = arrow_ipc.shape
a[0].compute(),a[1]

CPU times: user 45.7 s, sys: 3.74 s, total: 49.5 s
Wall time: 49.7 s


(11791325, 5)

In [ ]:
%%time

unmapped = arrow_ipc[pd.isna(arrow_ipc["rname"])].groupby("seq").agg("count")

In [ ]:
for row in arrow_ipc.itertuples():
    print(row)
    break

In [9]:
unmapped.groupby("seq").size()

seq
CGACTGAGTCCGCGATGGAGAGAGCTGTAGGCACCA    1
GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT    4
dtype: int64

In [25]:
%%time
unmapped = arrow_ipc.filter(pl.col("rname").is_null())
dct['unmapped_reads'] = unmapped.shape[0]
unmapped_seq_counts = unmapped.group_by("seq").len()
dct["frequent_unmapped_reads"] = unmapped_seq_counts.sort("len", descending=True).head(2000)["seq"].to_list()


CPU times: user 2.53 s, sys: 484 ms, total: 3.01 s
Wall time: 704 ms


In [23]:
max_seq_len = max(arrow_ipc["seq"].map_elements(len, return_dtype=int))

In [24]:
data = []
for seq in tqdm(unmapped_seq_counts.iter_rows(named=True)):
    seqlen = len(row["seq"])
    data.append(list(row["seq"])+[""]*(max_seq_len-seqlen)+[row["len"]])
nuc_dist = pl.DataFrame(data)
del data
nuc_dist.columns = nuc_dist.columns.map(str)
nuc_dist["length"] = seq_len

36

In [54]:
mapped = arrow_ipc.filter(pl.col("rname").is_not_null()).with_columns(pl.col("rname").map_elements(lambda x:x.split(".")[0], return_dtype=str))

In [14]:
transcriptome = pl.read_database(
    query="SELECT * FROM transcripts", 
    connection=conn,
).lazy().select(["transcript","cds_start","cds_stop","length","strand","chrom","tran_type"])#.filter(pl.col("tran_type")==1)

In [30]:
exons = (
    pl.read_database(
        query="SELECT * FROM exons", 
        connection=conn,
    )
    .lazy()
    .sort("exon_start")
    .with_columns(
        ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])
    )
    .group_by("transcript")
    .agg(pl.col("ranges"))
    .join(transcriptome, on="transcript")
    .with_columns(local_range = pl.struct("ranges").map_elements(lambda x: list(np.cumsum([0] + [y[1]-y[0]+1 for y in x["ranges"]])))# TODO: bring it back ,return_dtype=list
                 ).rename({"transcript":"rname"})
)

In [26]:
exons.head().collect()

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
/tmp/ipykernel_39093/335241348.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  exons.head().collect()


transcript,ranges,cds_start,cds_stop,length,strand,chrom,tran_type,local_range
str,list[list[i64]],i64,i64,i64,str,str,i64,list[i64]
"""ENST00000630264""","[[30377821, 30378143], [30379483, 30379623], [30379712, 30379781]]",1,93,534,"""-""","""chr22""",1,"[0, 323, … 534]"
"""ENST00000474746""","[[77581819, 77581889], [77581980, 77582127], … [77584910, 77585060]]",null,null,572,"""-""","""chr1""",0,"[0, 71, … 572]"
"""ENST00000389817""","[[17392886, 17393129], [17393698, 17393760], … [17476630, 17476846]]",70,4815,4921,"""-""","""chr11""",1,"[0, 244, … 4921]"
"""ENST00000606813""","[[56927874, 56929573]]",null,null,1700,"""+""","""chr5""",0,"[0, 1700]"
"""ENST00000525691""","[[113240507, 113240833], [113246368, 113246370], … [113260146, 113260275]]",null,null,585,"""+""","""chr11""",0,"[0, 327, … 585]"


In [51]:
qname_count = mapped.group_by("qname").len()#.rename({"qname":"qqname"})

In [37]:
mapped = mapped.join(qname_count, on="qname")

In [38]:
multi_mapped = mapped.filter(pl.col("len")>1)

In [12]:
single_mapped = mapped.filter(pl.col("len")==1)

In [39]:
def genomic_pos(x):
    # print(x)
    idx = np.where(np.array(x["local_range"]) < x["pos"])[0][-1]
    genomic_range = x["ranges"][idx]
    if x["strand"] == "+":
        return genomic_range[0] + x["pos"] - x["local_range"][0]
    else:
        return genomic_range[1] - ( x["pos"] - x["local_range"][0])

In [40]:
multi_mapped = (multi_mapped.
                join(exons, on="rname")
                .with_columns(genomic_pos = (pl.struct("pos","ranges","local_range","strand")
                                             .map_elements(lambda x: genomic_pos(x) ))
                              )
               ).drop("ranges","local_range")

In [55]:
mapped.columns

/tmp/ipykernel_39093/482220260.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  mapped.columns


['qname', 'rname', 'pos', 'seq', 'tags']

In [4]:
import polars as pl

In [5]:
df = pl.read_ipc(arrow_ipc)

In [22]:
df.filter(pl.col("qname")=="M01074:70:000000000-AAVBP:1:2118:8800:3666")

qname,flag,rname,pos,mapq,cigar,rnext,pnext,tlen,seq,qual,tags,end
str,u16,cat,i32,u8,str,cat,i32,i32,str,str,struct[6],i32
"""M01074:70:000000000-AAVBP:1:21…",147,"""NC_001802.1""",32,40,"""71M""","""NC_001802.1""",49,-54,"""GGAGCTCTCTGGCTACTTAGGGAACCCACT…","""GGGGGGGGGGGGGGGGGGGGGGGGGGGGGG…","{53,""15A0C46T0C6"",null,4,53,""NC_001802.1,-9117,63M8S,2;""}",102
"""M01074:70:000000000-AAVBP:1:21…",99,"""NC_001802.1""",49,60,"""2S69M""","""NC_001802.1""",32,54,"""CTTAGGGAACCCACTGCTTAAGCCTCAATA…","""GGGGGGGGGGGGGGGGGGGGGGGGGGGGGG…","{46,""46T0C21"",null,2,59,null}",117


In [21]:
df.filter(pl.col("qname")=="M01074:70:000000000-AAVBP:1:2118:8800:3666")['seq'].to_numpy()

array(['GGAGCTCTCTGGCTACTTAGGGAACCCACTGCTTAAGCCTCAATAAAGCTTGCCTTGAGTGCTCTAAGTAG',
       'CTTAGGGAACCCACTGCTTAAGCCTCAATAAAGCTTGCCTTGAGTGCTCTAAGTAGTGTGTGCCCGTCTGT'],
      dtype=object)

In [23]:
 ox.from_bam("RES020_S17_L001001_sequence-alignment.bam")

AttributeError: module 'oxbow' has no attribute 'from_bam'